In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# --- 1. DEFINE PIPELINE COMPONENT CLASSES DIRECTLY ---

class XenteFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.customer_aggs_ = {}

    def fit(self, X, y=None):
        df = X.copy()
        if 'CustomerId' in df.columns and 'Amount' in df.columns:
            group = df.groupby('CustomerId')['Amount']
            self.customer_aggs_ = {
                'total': group.sum().to_dict(),
                'avg': group.mean().to_dict(),
                'count': group.count().to_dict(),
                'std': group.std().fillna(0).to_dict()
            }
        return self

    def transform(self, X):
        df = X.copy()
        if 'CustomerId' in df.columns:
            df['Total_Amount_Cust'] = df['CustomerId'].map(self.customer_aggs_.get('total', {})).fillna(df['Amount'])
            df['Avg_Amount_Cust'] = df['CustomerId'].map(self.customer_aggs_.get('avg', {})).fillna(df['Amount'])
            df['Transaction_Count_Cust'] = df['CustomerId'].map(self.customer_aggs_.get('count', {})).fillna(1)
            df['Std_Amount_Cust'] = df['CustomerId'].map(self.customer_aggs_.get('std', {})).fillna(0)
        else:
            df['Total_Amount_Cust'] = df['Amount']
            df['Avg_Amount_Cust'] = df['Amount']
            df['Transaction_Count_Cust'] = 1
            df['Std_Amount_Cust'] = 0

        if 'TransactionStartTime' in df.columns:
            times = pd.to_datetime(df['TransactionStartTime'])
            df['TransactionHour'] = times.dt.hour
            df['TransactionDay'] = times.dt.day
            df['TransactionMonth'] = times.dt.month
            df['TransactionYear'] = times.dt.year
        else:
            df['TransactionHour'] = 0
            df['TransactionDay'] = 1
            df['TransactionMonth'] = 1
            df['TransactionYear'] = 2026

        drop_cols = ['TransactionId', 'BatchId', 'AccountId', 'SubscriptionId', 'CustomerId', 'TransactionStartTime']
        return df.drop(columns=[c for c in drop_cols if c in df.columns])


class NativeWoETransformer(BaseEstimator, TransformerMixin):
    """
    A robust, native Pandas Weight of Evidence (WoE) transformer.
    Replaces the broken xverse package dependency completely.
    """
    def __init__(self, bins=5):
        self.bins = bins
        self.woe_maps_ = {}
        self.iv_scores_ = {}

    def fit(self, X, y):
        if y is None:
            raise ValueError("WoE calculation requires an explicit binary target vector 'y'.")
        
        df = pd.DataFrame(X).copy()
        y_arr = np.array(y)
        total_pos = np.sum(y_arr == 1)
        total_neg = np.sum(y_arr == 0)

        # Smooth out any zero division issues
        total_pos = total_pos if total_pos > 0 else 1
        total_neg = total_neg if total_neg > 0 else 1

        for col in df.columns:
            # If numerical with more than 10 unique values, bin it first to form groups
            if pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() > 10:
                try:
                    series_binned = pd.qcut(df[col], q=self.bins, duplicates='drop').astype(str)
                except ValueError:
                    series_binned = pd.cut(df[col], bins=self.bins).astype(str)
            else:
                series_binned = df[col].astype(str)

            # Calculate event counts per group
            stats = pd.DataFrame({'feature': series_binned, 'target': y_arr})
            group_counts = stats.groupby('feature')['target'].agg(pos='sum', total='count')
            group_counts['neg'] = group_counts['total'] - group_counts['pos']

            # Apply Laplace smoothing to protect against 0 values log(0)
            group_counts['pos_dist'] = (group_counts['pos'] + 0.5) / total_pos
            group_counts['neg_dist'] = (group_counts['neg'] + 0.5) / total_neg

            # Compute Weight of Evidence and Information Value
            group_counts['woe'] = np.log(group_counts['pos_dist'] / group_counts['neg_dist'])
            group_counts['iv'] = (group_counts['pos_dist'] - group_counts['neg_dist']) * group_counts['woe']

            self.woe_maps_[col] = group_counts['woe'].to_dict()
            self.iv_scores_[col] = group_counts['iv'].sum()

        return self

    def transform(self, X):
        df = pd.DataFrame(X).copy()
        df_woe = pd.DataFrame(index=df.index)
        
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() > 10:
                try:
                    series_binned = pd.qcut(df[col], q=self.bins, duplicates='drop').astype(str)
                except ValueError:
                    series_binned = pd.cut(df[col], bins=self.bins).astype(str)
            else:
                series_binned = df[col].astype(str)

            # Map categories/bins directly to their computed WoE values
            df_woe[f'{col}_woe'] = series_binned.map(self.woe_maps_[col]).fillna(0)
            
        return df_woe


def build_feature_engineering_pipeline() -> Pipeline:
    numeric_features = ['Amount', 'Value', 'Total_Amount_Cust', 'Avg_Amount_Cust', 'Transaction_Count_Cust', 'Std_Amount_Cust']
    categorical_features = ['ProductCategory', 'ChannelId', 'ProviderId', 'ProductId', 'CurrencyCode']
    temporal_features = ['TransactionHour', 'TransactionDay', 'TransactionMonth', 'TransactionYear']
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
            ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features),
            ('time', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('scaler', StandardScaler())]), temporal_features)
        ],
        remainder='passthrough'
    )

    return Pipeline([
        ('feature_engineer', XenteFeatureEngineer()),
        ('preprocessor', preprocessor),
        ('woe_transform', NativeWoETransformer(bins=5))
    ])


# --- 2. DATA PROCESSING SEQUENCE EXECUTOR ---

raw_data_path = "../data/raw/data.csv"

if not os.path.exists(raw_data_path):
    raise FileNotFoundError(f"Dataset path could not be located: {raw_data_path}")

print("📖 Loading raw Xente dataset...")
df_raw = pd.read_csv(raw_data_path, sep=',', parse_dates=['TransactionStartTime'])

X = df_raw.drop(columns=['FraudResult'])
y = df_raw['FraudResult']

pipeline = build_feature_engineering_pipeline()

print("⚙️ Running feature engineering pipeline...")
X_model_ready = pipeline.fit_transform(X, y)

# Generate destination project subdirectories
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Package, merge targets, and commit the transformed matrix to disk
df_processed = pd.DataFrame(X_model_ready)
df_processed['Target_FraudResult'] = y.values

processed_data_path = "../data/processed/model_ready_xente.csv"
df_processed.to_csv(processed_data_path, index=False)
print(f"✅ Transformed dataset successfully exported: {processed_data_path}")

pipeline_path = "../models/feature_engineering_pipeline.pkl"
joblib.dump(pipeline, pipeline_path)
print(f"📦 Feature engineering pipeline schema compiled: {pipeline_path}")


📖 Loading raw Xente dataset...
⚙️ Running feature engineering pipeline...
✅ Transformed dataset successfully exported: ../data/processed/model_ready_xente.csv
📦 Feature engineering pipeline schema compiled: ../models/feature_engineering_pipeline.pkl


In [2]:
# Extract the native transformer step directly from your running pipeline object
woe_step = pipeline.named_steps['woe_transform']

# Convert the tracking dictionary into a clean sorted pandas DataFrame
iv_df = pd.DataFrame(list(woe_step.iv_scores_.items()), columns=['Feature', 'Information_Value'])
iv_df = iv_df.sort_values(by='Information_Value', ascending=False).reset_index(drop=True)

print("🏆 Engineered Feature Strength Ranked by Information Value (IV):")
print(iv_df)


🏆 Engineered Feature Strength Ranked by Information Value (IV):
    Feature  Information_Value
0         1           4.298839
1         0           4.194507
2         5           3.948213
3         3           3.725611
4         2           3.376075
5        31           2.360139
6        44           1.405034
7        22           1.164750
8        24           1.162410
9        16           1.113743
10       17           0.923232
11       54           0.832606
12       21           0.822340
13        6           0.804922
14        8           0.613784
15        4           0.542379
16       19           0.456663
17       41           0.305892
18       26           0.218848
19       23           0.185830
20       49           0.100915
21       51           0.096053
22       50           0.094112
23       47           0.086168
24       29           0.074310
25       25           0.068980
26       14           0.054269
27       42           0.052137
28       12           0.050418
29    